In [1]:
# ============================================================
# Notebook : 07_hitl_governance.ipynb
# Purpose  : Step 5 (Governance). Assign a human-in-the-loop (HITL)
#            oversight grade to each abnormal category from BOTH axes of
#            the prior steps - the Step-3 causal reproduction verdict and
#            the Step-4 model precision - using an explicit two-dimensional
#            grid with a conservative (more-restrictive-axis-wins) rule.
#            This repairs the earlier one-dimensional rule, under which a
#            category that only PARTIALLY reproduced could still receive the
#            top autonomy grade on the strength of its precision alone. The
#            one-dimensional grade is reported alongside the two-dimensional
#            grade to make the correction explicit. A fairness audit over
#            age, sex and region and a per-category case report are produced.
# Design   : Single code cell. No absolute path is printed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.unicode_minus": False,
    "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85",
})


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
repro = pd.read_pickle(DATA / "reproduction.pkl")
perf = pd.read_csv(TAB / "table_model_performance.csv").set_index("category")
ABN = meta["abnormal_categories"]
CAT_LABEL = {"HR": "Heart rate", "BR": "Breath rate", "ST": "Skin temp.",
             "SP": "SpO2", "ER": "Emergency"}

# ============================================================
# Two-dimensional HITL grid (repairs the earlier precision-only rule).
#   Grade meaning (as in the framework):
#     1 = autonomous recommendation allowed, post-hoc sample audit
#     2 = restricted (risk-signal only), every positive pre-reviewed
#     3 = not allowed (raw signal only), immediate alert + human first-check
#   Rule (both axes required for the top grade; the model-untrustworthy
#   or direct-signal case forces the floor):
#     grade 3 : precision < 0.10  OR  direct raw signal
#     grade 1 : causal reproduction FULL  AND  precision >= 0.40
#     grade 2 : otherwise
#   The key correction over the one-dimensional rule: FULL causal
#   reproduction is NECESSARY for grade 1, so a category that only
#   partially reproduces (or whose causal structure does not reproduce)
#   is capped at grade 2 however high its precision.
# ============================================================
CAUSAL_LABEL = {"full": "stable reproduction", "partial": "partial reproduction",
                "autoregressive_only": "self-lag only (no cross-precursor)",
                "not_reproduced": "not reproduced", "direct_signal": "direct raw signal"}
FULL = {"full"}  # only a fully reproduced causal basis qualifies for grade 1


def grade_2d(verdict, p):
    if verdict == "direct_signal" or pd.isna(p) or p < 0.10:
        return 3
    if verdict in FULL and p >= 0.40:
        return 1
    return 2


def grade_1d(p):
    """Earlier precision-only rule (ignores causal reproduction)."""
    if pd.isna(p) or p < 0.10:
        return 3
    if p >= 0.40:
        return 1
    return 2


rows = []
for c in ABN:
    verdict = repro.get(c, "not_reproduced")
    p = perf.loc[c, "precision"] if c in perf.index else np.nan
    p = float(p) if pd.notna(p) else np.nan
    final = grade_2d(verdict, p)
    one_dim = grade_1d(p)
    rows.append({
        "category": c, "label": CAT_LABEL[c],
        "causal_verdict": CAUSAL_LABEL.get(verdict, verdict),
        "precision": (round(p, 3) if p == p else "n/a"),
        "grade_1D_precision_only": one_dim, "grade_2D_corrected": final,
        "downgraded_by_2D": final > one_dim,
    })
assign = pd.DataFrame(rows)
assign.to_csv(TAB / "table_hitl_assignment.csv", index=False, encoding="utf-8-sig")

# Grid definition table.
grid_rows = []
for verdict, cl in [("full", "stable"), ("partial", "partial"), ("not_reproduced", "not reproduced")]:
    for pz, pl in [(0.5, ">=0.40"), (0.2, "0.10-0.40"), (0.05, "<0.10")]:
        grid_rows.append({"causal": cl, "precision": pl, "grade": grade_2d(verdict, pz)})
grid = pd.DataFrame(grid_rows)
grid.to_csv(TAB / "table_hitl_grid.csv", index=False, encoding="utf-8-sig")

# ============================================================
# Fairness audit (exploratory; small sample). Flag-rate disparity across
# age band, sex and region for the autonomous-eligible (grade-1) categories.
# ============================================================
df = df.copy()
df["age_band"] = pd.cut(df["Age"], [0, 69, 79, 200], labels=["<=60s", "70s", ">=80s"])
# Audit the categories that reach at least restricted use (grade <= 2);
# if none would be autonomous (grade 1), the audit still covers graded risk use.
audit_cats = [r["category"] for r in rows if r["grade_2D_corrected"] <= 2]
fair_rows = []
if audit_cats:
    flag = (df[["cat_" + c for c in audit_cats]].sum(axis=1) > 0).astype(int)
    for dim in ["age_band", "Gender", "Region"]:
        g = df.assign(flag=flag).groupby(dim)["flag"].mean()
        disp = float(g.max() / g.min()) if g.min() > 0 else float("inf")
        for lev, val in g.items():
            fair_rows.append({"dimension": dim, "level": str(lev),
                              "flag_rate": round(float(val), 4)})
        fair_rows.append({"dimension": dim, "level": "__disparity_ratio__",
                          "flag_rate": round(disp, 2)})
fair = pd.DataFrame(fair_rows)
fair.to_csv(TAB / "table_fairness_audit.csv", index=False, encoding="utf-8-sig")

# ============================================================
# Case report for the category whose grade is corrected downward
# (the governance analogue of the paper's worked case).
# ============================================================
downs = assign[assign["downgraded_by_2D"]]
case_cat = downs.iloc[0]["category"] if len(downs) else assign.sort_values("grade_2D_corrected").iloc[0]["category"]
cr = assign[assign["category"] == case_cat].iloc[0]
case_report = {
    "category": CAT_LABEL[case_cat],
    "causal_reproduction": cr["causal_verdict"],
    "model_precision": cr["precision"],
    "grade_precision_only_(flawed)": int(cr["grade_1D_precision_only"]),
    "grade_two_dimensional_(corrected)": int(cr["grade_2D_corrected"]),
    "oversight": {1: "autonomous recommendation + post-hoc audit",
                  2: "restricted: risk-signal only, every positive pre-reviewed",
                  3: "raw signal only, immediate alert + human first-check"}[int(cr["grade_2D_corrected"])],
    "basis": "downgraded because the causal axis did not fully reproduce, "
             "even though precision alone would have granted a higher grade",
}
pd.to_pickle(case_report, DATA / "case_report.pkl")

# ============================================================
# Figure 12. HITL grade: one-dimensional vs two-dimensional, per category.
# ============================================================
fig, ax = plt.subplots(figsize=(6.4, 3.6))
x = np.arange(len(assign))
w = 0.38
ax.bar(x - w / 2, assign["grade_1D_precision_only"], w, label="1D (precision only)",
       color="0.75", edgecolor="black", linewidth=0.6)
ax.bar(x + w / 2, assign["grade_2D_corrected"], w, label="2D (causal + precision)",
       color="0.25", edgecolor="black", linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(assign["label"])
ax.set_ylabel("HITL grade (1 = most autonomy)")
ax.set_yticks([1, 2, 3])
ax.invert_yaxis()
ax.legend(frameon=False)
save_fig(fig, "fig12_hitl_grade_correction")

# ============================================================
# Figure 13. The two-dimensional grade grid (grayscale heatmap).
# ============================================================
G = grid.pivot(index="causal", columns="precision", values="grade")
G = G.reindex(index=["stable", "partial", "not reproduced"],
              columns=[">=0.40", "0.10-0.40", "<0.10"])
fig, ax = plt.subplots(figsize=(4.8, 3.6))
sns.heatmap(G, annot=True, fmt="d", cmap="Greys", vmin=1, vmax=3, linewidths=0.6,
            linecolor="0.5", cbar_kws={"label": "HITL grade"}, annot_kws={"size": 11}, ax=ax)
ax.set_xlabel("Model precision")
ax.set_ylabel("Causal reproduction")
save_fig(fig, "fig13_hitl_grid")

# ---- Console summary ----
print("=== Step 5: HITL governance (two-dimensional grade) ===")
print("Grade rule: grade 1 needs FULL causal reproduction AND precision >= 0.40;")
print("            grade 3 if precision < 0.10 or direct raw signal; otherwise grade 2.")
print()
print(assign.to_string(index=False))
print()
n_down = int(assign["downgraded_by_2D"].sum())
print(f"Categories whose grade the 2D rule corrects downward vs precision-only: {n_down}")
for r in rows:
    if r["downgraded_by_2D"]:
        print(f"  {r['label']}: precision-only grade {r['grade_1D_precision_only']} "
              f"-> corrected grade {r['grade_2D_corrected']}  ({r['causal_verdict']})")
print()
n_g1 = int((assign["grade_2D_corrected"] == 1).sum())
print(f"Categories eligible for autonomous use (grade 1) under the 2D rule: {n_g1} of {len(ABN)}")
print()
print(f"Case report category: {case_report['category']}  "
      f"-> grade {case_report['grade_two_dimensional_(corrected)']} "
      f"({case_report['oversight']})")
print()
print("Fairness audit (flag-rate disparity, exploratory):")
print(fair.to_string(index=False))
print()
print("Figures saved: fig12_hitl_grade_correction, fig13_hitl_grid  (png + pdf, 600 dpi)")
print("Tables saved : table_hitl_assignment, table_hitl_grid, table_fairness_audit")


=== Step 5: HITL governance (two-dimensional grade) ===
Grade rule: grade 1 needs FULL causal reproduction AND precision >= 0.40;
            grade 3 if precision < 0.10 or direct raw signal; otherwise grade 2.

category       label                     causal_verdict  precision  grade_1D_precision_only  grade_2D_corrected  downgraded_by_2D
      HR  Heart rate                     not reproduced      0.963                        1                   2              True
      BR Breath rate                     not reproduced      0.921                        1                   2              True
      ST  Skin temp.                stable reproduction      0.288                        2                   2             False
      SP        SpO2 self-lag only (no cross-precursor)      0.760                        1                   2              True
      ER   Emergency                  direct raw signal      0.000                        3                   3             False

Categor